# Stage 5 — Testing the Model

Questions:
- How well does the model rank customers it has never seen?
- What threshold should we use?
- What drives the scores?

**Rule:** pick the threshold using training data only, then test once. No changes after seeing the results.

The data is a snapshot, so this isn't a forecast of next month. Coefficients show links, not causes.

## Setup

Uses the same split and model as `03_modeling.ipynb`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, average_precision_score,
                             brier_score_loss, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import cross_val_predict

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from features import ID_COLUMN, load_modelling_data, split
from modeling import CV
from train_model import build_model

data = load_modelling_data()
X_train, X_test, y_train, y_test = split(data)

7,032 customers for modelling (11 with tenure = 0 excluded)
train 5,274 rows, churn 0.266
test  1,758 rows, churn 0.266   (untouched until Stage 5)


## Pick a threshold (training data only)

Each training customer gets a score from a model that didn't see them. We pick the threshold with the best F1. It's a starting point; the business may value catching churners and avoiding wasted contacts differently.

In [2]:
oof_probabilities = cross_val_predict(
    build_model(), X_train, y_train, cv=CV, method="predict_proba"
)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, oof_probabilities)
oof_f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_index = np.flatnonzero(oof_f1 == oof_f1.max())[-1]
selected_threshold = float(thresholds[best_index])

print(f"OOF-selected threshold: {selected_threshold:.4f}")
print(f"OOF precision: {precision[best_index]:.3f}")
print(f"OOF recall:    {recall[best_index]:.3f}")
print(f"OOF F1:        {oof_f1[best_index]:.3f}")

OOF-selected threshold: 0.3187
OOF precision: 0.556
OOF recall:    0.740
OOF F1:        0.635


## Test once

0.5 is shown for comparison. Don't change the threshold based on these results.

In [3]:
model = build_model().fit(X_train, y_train)
test_probabilities = model.predict_proba(X_test)[:, 1]
test_actual = y_test.to_numpy()

def operating_metrics(actual, predicted):
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
    return {
        "customers_flagged": int(predicted.sum()),
        "share_flagged": predicted.mean(),
        "precision": precision_score(actual, predicted, zero_division=0),
        "recall": recall_score(actual, predicted, zero_division=0),
        "f1": f1_score(actual, predicted, zero_division=0),
        "accuracy": accuracy_score(actual, predicted),
        "specificity": tn / (tn + fp),
        "true_positives": int(tp),
        "false_positives": int(fp),
        "false_negatives": int(fn),
    }

test_operating_points = pd.DataFrame({
    "threshold 0.50": operating_metrics(test_actual, (test_probabilities >= 0.5).astype(int)),
    "OOF-selected threshold": operating_metrics(
        test_actual, (test_probabilities >= selected_threshold).astype(int)),
}).T
test_operating_points.round(3)

,customers_flagged,share_flagged,precision,recall,f1,accuracy,specificity,true_positives,false_positives,false_negatives
threshold 0.50,408.0,0.232,0.654,0.572,0.610,0.806,0.891,267.0,141.0,200.0
OOF-selected threshold,639.0,0.363,0.543,0.743,0.627,0.766,0.774,347.0,292.0,120.0


In [4]:
base_rate = float(y_train.mean())
test_quality = pd.Series({
    "ROC-AUC": roc_auc_score(test_actual, test_probabilities),
    "PR-AUC (average precision)": average_precision_score(test_actual, test_probabilities),
    "Brier score": brier_score_loss(test_actual, test_probabilities),
    "Base-rate Brier baseline": brier_score_loss(
        test_actual, np.full(len(test_actual), base_rate)),
}, name="held-out test")
test_quality.round(4)

ROC-AUC                       0.8386
PR-AUC (average precision)    0.6231
Brier score                   0.1386
Base-rate Brier baseline      0.1951
Name: held-out test, dtype: float64

- **ROC-AUC / PR-AUC:** how well the model ranks customers.
- **Brier score:** how accurate the probabilities are (lower is better). Compare it with always guessing the average.

## If we can only contact some customers

Rank by score and check the top 10%, 20% and 30%. These are examples, not targets.

In [5]:
def capacity_metrics(actual, probabilities, share):
    count = int(np.ceil(len(actual) * share))
    selected = np.argsort(-probabilities, kind="stable")[:count]
    churners_found = int(actual[selected].sum())
    return {
        "customers_contacted": count,
        "precision": churners_found / count,
        "recall": churners_found / int(actual.sum()),
        "churners_captured": churners_found,
    }

capacity_results = pd.DataFrame({
    f"top {share:.0%}": capacity_metrics(test_actual, test_probabilities, share)
    for share in (0.10, 0.20, 0.30)
}).T
capacity_results.round(3)

,customers_contacted,precision,recall,churners_captured
top 10%,176.0,0.699,0.263,123.0
top 20%,352.0,0.662,0.499,233.0
top 30%,528.0,0.598,0.677,316.0


## Are the probabilities accurate?

Compare predicted and actual churn in each score group. Small groups can be noisy.

In [6]:
calibration = pd.DataFrame({
    "probability": test_probabilities,
    "actual": test_actual,
})
calibration["decile"] = pd.qcut(calibration["probability"], 10, duplicates="drop")
calibration = calibration.groupby("decile", observed=True).agg(
    customers=("actual", "size"),
    mean_probability=("probability", "mean"),
    observed_churn_rate=("actual", "mean"),
)
calibration["gap_percentage_points"] = (
    100 * (calibration["observed_churn_rate"] - calibration["mean_probability"])
)
calibration.round(3)

,customers,mean_probability,observed_churn_rate,gap_percentage_points
decile,,,,
"(0.00286, 0.0139]",176,0.009,0.017,0.815
"(0.0139, 0.0316]",176,0.022,0.011,-1.031
"(0.0316, 0.0638]",176,0.046,0.051,0.535
"(0.0638, 0.125]",175,0.091,0.091,0.016
"(0.125, 0.188]",176,0.158,0.205,4.652
"(0.188, 0.284]",176,0.231,0.222,-0.987
"(0.284, 0.414]",175,0.343,0.263,-7.988
"(0.414, 0.538]",176,0.478,0.472,-0.640
"(0.538, 0.667]",176,0.609,0.625,1.617


## What drives the score

Odds ratios above 1 raise the score; below 1 lower it. Each is compared to that variable's base category. These explain the model, they don't show causes.

In [7]:
coefficients = model.named_steps["model"].coef_[0]
feature_names = model.named_steps["prep"].get_feature_names_out()
effects = pd.DataFrame({"feature": feature_names, "log_odds": coefficients})
effects["odds_ratio"] = np.exp(effects["log_odds"])

print("Largest positive coefficients")
display(effects.nlargest(10, "log_odds").round(3))
print("Largest negative coefficients")
display(effects.nsmallest(10, "log_odds").round(3))

Largest positive coefficients


,feature,log_odds,odds_ratio
6,categorical__InternetService_Fiber optic,0.915,2.496
18,categorical__PaymentMethod_Electronic check,0.344,1.411
5,categorical__MultipleLines_Yes,0.340,1.404
16,categorical__PaperlessBilling_Yes,0.328,1.388
12,categorical__StreamingTV_Yes,0.315,1.370
13,categorical__StreamingMovies_Yes,0.273,1.314
1,categorical__SeniorCitizen_1,0.197,1.217
19,categorical__PaymentMethod_Mailed check,0.132,1.141
10,categorical__DeviceProtection_Yes,0.029,1.030
2,categorical__Partner_Yes,0.011,1.011


Largest negative coefficients


,feature,log_odds,odds_ratio
15,categorical__Contract_Two year,-1.307,0.271
7,categorical__InternetService_No,-0.866,0.421
20,numeric__tenure,-0.805,0.447
14,categorical__Contract_One year,-0.678,0.507
4,categorical__PhoneService_Yes,-0.509,0.601
8,categorical__OnlineSecurity_Yes,-0.409,0.664
11,categorical__TechSupport_Yes,-0.406,0.666
3,categorical__Dependents_Yes,-0.256,0.774
9,categorical__OnlineBackup_Yes,-0.170,0.844
0,categorical__gender_Male,-0.037,0.964


## Save test scores

Customer IDs are kept for checking, not used by the model. This is the test set only, so it's not a contact list.

In [8]:
test_predictions = pd.DataFrame({
    ID_COLUMN: data.loc[X_test.index, ID_COLUMN].to_numpy(),
    "churn_actual": test_actual,
    "churn_probability": test_probabilities,
    "flagged_at_oof_f1_threshold": (test_probabilities >= selected_threshold).astype(int),
    "oof_f1_threshold": selected_threshold,
})
prediction_path = ROOT / "data" / "processed" / "churn_test_predictions.csv"
prediction_path.parent.mkdir(parents=True, exist_ok=True)
test_predictions.sort_values("churn_probability", ascending=False).to_csv(
    prediction_path, index=False)
print(f"Saved {len(test_predictions):,} held-out scores to {prediction_path.relative_to(ROOT)}")

Saved 1,758 held-out scores to data/processed/churn_test_predictions.csv


## Results

- **ROC-AUC 0.839** on the test set vs 0.844 in cross-validation. No overfitting.
- **Threshold 0.32** catches 74% of churners vs 57% at 0.5, but contacts more customers who would have stayed.
- **Top 20%** of scores holds half of all churners.
- **Probabilities match** actual churn within 5 points in 9 of 10 groups.
- **Streaming and multiple lines** raise the score because they cost extra. They stand in for price, not churn causes.

## Next

- Don't change the model or threshold based on these results.
- The final threshold depends on how many customers the team can contact and the cost of a missed churner vs a wasted contact.
- Stage 6 combines scores with customer value to build the priority list.
- The data can't show future churn or whether offers work.